# 02 · Stage 2 — disease-group router (multi-label)

Trained on unhealthy images only. Seven sigmoid outputs: six clinical groups
(vascular, exudative, optic disc, structural, media, inflammatory) plus the pooled
`G_Rare` flag. An image is routed to **every** group that fires.

`LOSS` / `USE_GEM` reproduce the Stage-2 ablation (BCE, focal, ASL, GeM pooling);
BCE without GeM was selected and is used by the cascade.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # or: pip install -e ..

import numpy as np, pandas as pd, torch
from fundus import config, labels
from fundus.data import (load_split, unhealthy_only, FundusDataset, make_loader,
                         train_transform, eval_transform, pos_weight)
from fundus.engine import set_seed, device, fit, predict

P = config.Paths()          # reads $FUNDUS_DATA_ROOT and $FUNDUS_RUN_ROOT
set_seed(config.SEED)
DEV = device()
print("device:", DEV, "| data:", P.data_root.resolve())

In [ ]:
LOSS, USE_GEM = "bce", False          # ablation: "bce" | "focal" | "asl", GeM on/off
cfg = config.STAGE2
G = labels.GROUP_COLS

tr = unhealthy_only(load_split(P.csv("train")))
va = unhealthy_only(load_split(P.csv("val")))
print("train images per number of groups:", tr[G].sum(1).value_counts().sort_index().to_dict())
print(tr[G].sum().to_string())

train_ds = FundusDataset(tr, P.images("train"), train_transform(cfg.img_size), label_cols=G)
val_ds   = FundusDataset(va, P.images("val"),   eval_transform(cfg.img_size),  label_cols=G)
train_dl = make_loader(train_ds, cfg.batch_size, True,  cfg.num_workers)
val_dl   = make_loader(val_ds,   cfg.batch_size, False, cfg.num_workers)

In [ ]:
from fundus.models import build_classifier, GeMClassifier
from fundus.losses import make_criterion
from fundus.engine import evaluate_multilabel

model = GeMClassifier(len(G), cfg) if USE_GEM else build_classifier(len(G), cfg)
criterion = make_criterion(LOSS, pos_weight(train_ds.labels, cfg.pos_weight_clip))
OUT = P.run(f"stage2_{LOSS}{'_gem' if USE_GEM else ''}")
hist = fit(model, train_dl, val_dl, criterion, evaluate_multilabel, OUT, cfg, monitors=("macro_auc",))

## Per-group diagnostics and routing thresholds (validation)

In [ ]:
from fundus.models import load_weights
from fundus.metrics import best_f1_threshold
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

model = load_weights(model, OUT / "best_macro_auc.pt", DEV)
_, (Y, Pv, S) = evaluate_multilabel(model, val_dl, DEV)

thr, rows = {}, []
for j, g in enumerate(G):
    thr[g] = best_f1_threshold(Y[:, j], Pv[:, j])
    rows.append({"group": g, "n_val": int(Y[:, j].sum()),
                 "auc": roc_auc_score(Y[:, j], Pv[:, j]), "ap": average_precision_score(Y[:, j], Pv[:, j]),
                 "f1@thr": f1_score(Y[:, j], (Pv[:, j] >= thr[g]).astype(int)), "thr": thr[g]})
tab = pd.DataFrame(rows).set_index("group")
print(tab.round(3).to_string())
clin = [g for g in G if g != labels.RARE_COL]
print(f"6-group macro-AUC {tab.loc[clin, 'auc'].mean():.4f} | macro-AP {tab.loc[clin, 'ap'].mean():.4f}")
json.dump({"thresholds": thr}, open(OUT / "operating_points.json", "w"), indent=2)